# Sigma1 Experiment Count Convergence

How many compact sequences are needed for stable sigma1 fits?

This notebook fixes sigma0 from Stage A, generates a large pool of
high-diversity compact sequences, then runs `fit_sigma_1d` for sigma1
with increasing numbers of randomly-sampled sequences. All fits use
the **sparse 12-point AUROC** method (`compute_auroc_sparse12`).

The goal: identify the minimum number of sequences where best-fit
sigma1 stabilizes and the MSE landscape is well-resolved.

| Sweep parameter | Values |
|-----------------|--------|
| `N_SEQS` | 3, 5, 8, 10, 15, 20, 30 |
| `n_seqs_per_rep` | N (all sequences used each MC rep) |
| `N_MC` | 8 |
| `N_GRID` | 10 |
| AUROC method | sparse 12-point |

Additionally, we investigate how the number of FPR sampling points
(`n_points`) in `compute_auroc_sparse` affects the d' vs sigma1
landscape — trading off between noise suppression (fewer points)
and AUROC accuracy (more points).

In [ ]:
import sys, os, yaml, torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter
from scipy.spatial.distance import pdist

sys.path.append('/om2/user/jmhicks/projects/TextureStreaming/code/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/utls/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/src/model/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/')

from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params, statistics_set, texture_dataset
from texture_prior.utils import path

from utls.plotting import ensure_dir
from utls.loading import (
    load_results_with_exclusion_2,
    move_sequences_to_used,
    load_results_with_exclusion_no_dropping,
)
from utls.runners_v2 import run_experiment_scores, make_noise_schedule
from utls.runners_utils import *
from utls.analysis_helpers import auroc_to_dprime
from utls.io_utils import (
    make_model_save_dir,
    save_all_figures,
    save_single_figure,
    save_runs_summary,
)
from encoders import *

from utls.toy_experiments import (
    make_toy_experiment_list,
    make_compact_multi_isi_sequences,
    infer_trial_isis,
    make_high_diversity_sequences
)
from utls.sigma_fitting import (
    log_mid,
    fit_sigma_1d,
    plot_sigma_fit,
    compute_auroc_sparse,
    compute_auroc_sparse12,
)

## 1. Load config & data

In [ ]:
def load_config(cfg_path):
    cfg_path = Path(cfg_path)
    if not cfg_path.exists():
        raise FileNotFoundError(cfg_path)
    with open(cfg_path) as f:
        return yaml.safe_load(f), cfg_path


CONFIG_PATH = (
    "/om2/user/bjmedina/auditory-memory/memory/"
    "model_yamls/three-regime/resnet50/nontime_avg/run_000005.yaml"
)

model_cfg, model_cfg_path = load_config(CONFIG_PATH)
print(model_cfg)

In [ ]:
# ---- experiment ----
exp_cfg = model_cfg["experiment"]
which_task = exp_cfg["which_task"]
is_multi = exp_cfg["is_multi"]
which_isi = exp_cfg.get("which_isi", None)

isis = [0, 1, 2, 4, 8, 16, 32, 64] if is_multi else [0, which_isi]

# ---- metric ----
metric = model_cfg["metric"]

# ---- noise model ----
noise_cfg = model_cfg["noise_model"]
noise_mode = noise_cfg["name"]
t_step = noise_cfg["t_step"]

# ---- representation ----
repr_cfg = model_cfg["representation"]
time_avg = repr_cfg["time_avg"]
encoder_type = repr_cfg["type"]
layer = repr_cfg.get("layer", None)
pc_dims = repr_cfg.get("pc_dims", None)

# ---- load human data ----
exp_list, all_files, name_to_idx, human_runs, task_name, hr_task_name = (
    load_experiment_data(which_task, which_isi, is_multi, old=False)
)

human_curve = compute_human_curve(human_runs, is_multi, which_isi)
print("ISIs:", isis)
print("Human d' curve:", human_curve)
print(f"Total real sequences: {len(exp_list)}")

## 2. Build encoder & encode stimuli

In [ ]:
NN_ENCODERS = {"kell2018", "resnet50"}
encoder_task = (
    "word_speaker_audioset" if encoder_type in NN_ENCODERS else "audioset"
)

encoder_cfg = dict(
    encoder_type=encoder_type,
    model_name=encoder_type,
    task=encoder_task,
    statistics_dict=statistics_set.statistics,
    model_params=model_params,
    pc_dims=pc_dims,
    sr=20000,
    duration=2.0,
    rms_level=0.05,
    time_avg=time_avg,
    device="cuda",
)

if encoder_type in NN_ENCODERS:
    encoder_cfg["layer"] = layer
if encoder_type == "texture":
    encoder_cfg["pc_dims"] = pc_dims

encoder_name = make_encoder_name(encoder_cfg)
print("Encoder name:", encoder_name)

encoder = build_encoder(encoder_cfg)
X = encode_stimuli(encoder, all_files)
X_np = X.detach().cpu().numpy()
print("Encoded shape:", X_np.shape, "  any NaN?", torch.isnan(X).any().item())

## 3. Parameter bounds & stimulus pool

In [ ]:
def median_pairwise_distance(X, metric="euclidean", n_samples=500, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.choice(X.shape[0], size=min(n_samples, X.shape[0]), replace=False)
    return float(np.median(pdist(X[idx], metric=metric)))


d50 = 1#median_pairwise_distance(X_np, metric="cosine")
print(f"Median pairwise cosine distance: {d50:.6f}")

param_bounds = {
    "sigma0": (
        0.0001,
        22,
    ),
    "sigma1": (
        0.0001,
        10,
    ),
    "sigma2": (
        0.0001,
        5,
    ),
}

print("Parameter bounds:")
for k, v in param_bounds.items():
    print(f"  {k}: ({v[0]:.6f}, {v[1]:.6f})")

# Stimulus pool for sequence generation
stimulus_pool = sorted({s for seq in exp_list for s in seq})
print(f"\nStimulus pool size: {len(stimulus_pool)}")
assert len(stimulus_pool) >= 65, (
    f"Stimulus pool ({len(stimulus_pool)}) too small for ISI-64 blocks (need >= 65)"
)

## 4. Human d' targets

In [ ]:
isi_to_hc_idx = {isi_val: i for i, isi_val in enumerate(isis)}

sigma0_human = {0: float(human_curve[isi_to_hc_idx[0]])}
sigma1_human = {isi: float(human_curve[isi_to_hc_idx[isi]]) for isi in [1, 2, 4]}
sigma2_human = {isi: float(human_curve[isi_to_hc_idx[isi]]) for isi in [8, 16, 32, 64]}

print("Stage A targets (sigma0):")
for isi, dp in sigma0_human.items():
    print(f"  ISI {isi}: human d' = {dp:.4f}")

print("\nStage B targets (sigma1):")
for isi, dp in sigma1_human.items():
    print(f"  ISI {isi}: human d' = {dp:.4f}")

print("\nStage C targets (sigma2):")
for isi, dp in sigma2_human.items():
    print(f"  ISI {isi}: human d' = {dp:.4f}")

# Initial values for unfitted sigmas
sigma1_init = log_mid(*param_bounds["sigma1"])
sigma2_init = log_mid(*param_bounds["sigma2"])
print(f"\nInitial sigma1 = {sigma1_init:.6f}")
print(f"Initial sigma2 = {sigma2_init:.6f}")

---
## Stage A: Fit sigma0 (ISI = 0, toy experiments)

ISI=0 means immediate repeats — only one step of noise, so only sigma0
matters.  Standard toy experiments (pairs of identical stimuli) are
efficient here.

In [ ]:
isi0_exps = {
    0: make_toy_experiment_list(
        stimulus_pool, isi=0, n_experiments=100, k_stimuli=5, seed=0
    )
}
print(f"ISI-0 toy experiments: {len(isi0_exps[0])} exps, "
      f"avg len {np.mean([len(e) for e in isi0_exps[0]]):.0f} trials")

N_REFINE_ITERS = 2
N_MC = 8

stage_a = fit_sigma_1d(
    run_experiment_fn=run_experiment_scores,
    sigma_name="sigma0",
    sigma_bounds=param_bounds["sigma0"],
    fixed_sigmas={"sigma1": sigma1_init, "sigma2": sigma2_init},
    noise_mode=noise_mode,
    metric=metric,
    X0=X,
    name_to_idx=name_to_idx,
    experiments_by_isi=isi0_exps,
    human_dprimes_by_isi=sigma0_human,
    t_step=t_step,
    n_grid=15,
    n_mc=N_MC,
    n_refine_iters=N_REFINE_ITERS,
    seed=0,
    auroc_fn=compute_auroc_sparse12,
)

sigma0_fitted = stage_a["best_sigma"]
print(f"\n>>> sigma0 = {sigma0_fitted:.6f}  (MSE = {stage_a['best_mse']:.6f})")
plot_sigma_fit(stage_a, human_dprimes_by_isi=sigma0_human,
               title="Stage A: sigma0 (ISI = 0)");

In [ ]:
all_stimuli = set()

for i in range(len(isi0_exps[0])):
    all_stimuli = all_stimuli.union(set(isi0_exps[0][i]))

print(len(all_stimuli))

---
## Stage B: Sigma1 experiment count convergence

Generate a large pool of sequences, then sweep over how many are used
for the sigma1 fit.

In [ ]:
SIGMA1_ISIS = [1, 2, 4]
SIGMA1_LENGTH = 60
SIGMA1_MIN_PAIRS = 5
POOL_SIZE = 40  # large pool to sample from

sigma1_all_exps, sigma1_all_isi_keys = make_high_diversity_sequences(
    stimulus_pool=stimulus_pool,
    isi_values=SIGMA1_ISIS,
    n_sequences=POOL_SIZE,
    length=SIGMA1_LENGTH,
    min_pairs_per_isi=SIGMA1_MIN_PAIRS,
    seed=1000,
)

print(f"Generated {len(sigma1_all_exps)} sequences, {len(sigma1_all_exps[0])} trials each")

# Validate ISI distribution
from collections import defaultdict
isi_counts = defaultdict(list)
for seq in sigma1_all_exps:
    counts = Counter(infer_trial_isis(seq))
    for isi_val in SIGMA1_ISIS:
        isi_counts[isi_val].append(counts.get(isi_val, 0))

print("\nPairs per ISI per sequence (mean +/- std):")
for isi_val in SIGMA1_ISIS:
    vals = isi_counts[isi_val]
    print(f"  ISI {isi_val}: {np.mean(vals):.1f} +/- {np.std(vals):.1f}  "
          f"(min={min(vals)}, max={max(vals)})")

In [ ]:
N_SEQS_SWEEP = [3, 5, 8, 10, 15, 20, 30]
N_MC = 8
N_REFINE_ITERS = 2
N_GRID = 10

print(f"Sweep: {len(N_SEQS_SWEEP)} experiment counts")
print(f"N_SEQS values: {N_SEQS_SWEEP}")
print(f"N_MC={N_MC}, N_GRID={N_GRID}, N_REFINE_ITERS={N_REFINE_ITERS}")
print(f"AUROC method: compute_auroc_sparse12")

In [ ]:
fit_by_n_seqs = {}

for N in N_SEQS_SWEEP:
    print(f"\n{'='*60}")
    print(f"  N_SEQS = {N}  (n_seqs_per_rep = {N})")
    print(f"{'='*60}")

    # Randomly sample N sequences from the pool
    rng = np.random.default_rng(seed=42 + N)
    indices = rng.choice(len(sigma1_all_exps), size=N, replace=False)
    exps_subset = [sigma1_all_exps[i] for i in indices]
    isi_keys_subset = [sigma1_all_isi_keys[i] for i in indices]

    result = fit_sigma_1d(
        run_experiment_fn=run_experiment_scores,
        sigma_name="sigma1",
        sigma_bounds=param_bounds["sigma1"],
        fixed_sigmas={"sigma0": sigma0_fitted, "sigma2": sigma2_init},
        noise_mode=noise_mode,
        metric=metric,
        X0=X,
        name_to_idx=name_to_idx,
        human_dprimes_by_isi=sigma1_human,
        t_step=t_step,
        n_grid=N_GRID,
        n_mc=N_MC,
        n_refine_iters=N_REFINE_ITERS,
        seed=100_000 + N,
        experiment_list=exps_subset,
        isi_keys=isi_keys_subset,
        target_isis=SIGMA1_ISIS,
        n_seqs_per_rep=N,
        auroc_fn=compute_auroc_sparse12,
    )

    fit_by_n_seqs[N] = result
    print(f"  >>> best sigma1 = {result['best_sigma']:.6f}  (MSE = {result['best_mse']:.6f})")

print("\nAll sweeps complete.")

---
## Convergence Plots

In [ ]:
import matplotlib.cm as cm

fig, axes = plt.subplots(1, len(SIGMA1_ISIS), figsize=(6 * len(SIGMA1_ISIS), 5),
                         sharey=True)

colors = cm.viridis(np.linspace(0.15, 0.95, len(N_SEQS_SWEEP)))

for ax, isi_val in zip(axes, SIGMA1_ISIS):
    for color, N in zip(colors, N_SEQS_SWEEP):
        results = fit_by_n_seqs[N]["all_results"]
        sigma_vals = np.array([r["sigma_value"] for r in results])
        dp_key = "dprime_mean_by_isi" if "dprime_mean_by_isi" in results[0] else "dprime_by_isi"
        dp_vals = np.array([r[dp_key].get(isi_val, np.nan) for r in results])

        order = np.argsort(sigma_vals)
        ax.plot(sigma_vals[order], dp_vals[order], "o-", ms=3, color=color,
                label=f"N={N}", alpha=0.8)

    ax.axhline(sigma1_human[isi_val], ls=":", color="red", alpha=0.7, label="human d'")
    ax.set_xscale("log")
    ax.set_xlabel("sigma1")
    ax.set_title(f"ISI = {isi_val}")
    ax.grid(alpha=0.25)
    ax.legend(fontsize=7, frameon=False, ncol=2)

axes[0].set_ylabel("d'")
fig.suptitle("d' vs sigma1 — varying number of sequences", y=1.03, fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

n_vals = np.array(N_SEQS_SWEEP)
best_sigmas = np.array([fit_by_n_seqs[N]["best_sigma"] for N in N_SEQS_SWEEP])
best_mses = np.array([fit_by_n_seqs[N]["best_mse"] for N in N_SEQS_SWEEP])

# Left: best sigma1 vs N
ax = axes[0]
ax.plot(n_vals, best_sigmas, "o-", ms=6, color="tab:blue")
ax.set_xlabel("Number of sequences")
ax.set_ylabel("Best-fit sigma1")
ax.set_title("Best sigma1 vs experiment count")
ax.grid(alpha=0.25)

# Reference band: +/- 5% of the N=30 value
ref = best_sigmas[-1]
ax.axhspan(ref * 0.95, ref * 1.05, alpha=0.15, color="green", label=f"\u00b15% of N=30 ({ref:.4f})")
ax.legend(frameon=False, fontsize=8)

# Right: best MSE vs N
ax = axes[1]
ax.plot(n_vals, best_mses, "s-", ms=6, color="tab:orange")
ax.set_xlabel("Number of sequences")
ax.set_ylabel("Best MSE")
ax.set_title("Best MSE vs experiment count")
ax.grid(alpha=0.25)

fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

widths = []
for N in N_SEQS_SWEEP:
    results = fit_by_n_seqs[N]["all_results"]
    sigma_vals = np.array([r["sigma_value"] for r in results])
    mse_vals = np.array([r["mse_mean"] for r in results])

    best_mse = np.nanmin(mse_vals)
    threshold = 2 * best_mse
    within = sigma_vals[mse_vals <= threshold]
    width = np.log10(within.max() / within.min()) if len(within) > 1 else 0
    widths.append(width)

ax.plot(N_SEQS_SWEEP, widths, "o-", ms=6, color="tab:purple")
ax.set_xlabel("Number of sequences")
ax.set_ylabel("MSE minimum width (log10 ratio)")
ax.set_title("MSE landscape sharpness vs experiment count")
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

In [ ]:
print(f"{'N_SEQS':>7s} | {'best_sigma1':>12s} | {'best_MSE':>10s} | {'landscape_width':>16s}")
print("-" * 55)

ref_sigma = fit_by_n_seqs[N_SEQS_SWEEP[-1]]["best_sigma"]

for i, N in enumerate(N_SEQS_SWEEP):
    sigma = fit_by_n_seqs[N]["best_sigma"]
    mse = fit_by_n_seqs[N]["best_mse"]
    pct_diff = abs(sigma - ref_sigma) / ref_sigma * 100

    results = fit_by_n_seqs[N]["all_results"]
    sigma_vals = np.array([r["sigma_value"] for r in results])
    mse_vals = np.array([r["mse_mean"] for r in results])
    within = sigma_vals[mse_vals <= 2 * np.nanmin(mse_vals)]
    width = np.log10(within.max() / within.min()) if len(within) > 1 else 0

    marker = " <-- converged" if pct_diff < 5 else ""
    print(f"{N:>7d} | {sigma:>12.6f} | {mse:>10.6f} | {width:>16.3f}{marker}")

print(f"\nReference: N={N_SEQS_SWEEP[-1]}, sigma1={ref_sigma:.6f}")
print("Convergence criterion: <5% deviation from reference")

---
## Effect of n_points on AUROC and d'

How does the number of FPR sampling points in `compute_auroc_sparse`
affect the d' vs sigma1 curve? We reuse the fit results from the
N=30 case and recompute d' with varying n_points.

Note: this does NOT re-run the model — it only recomputes AUROC from
the same raw hit/FA scores. To do this we need a sweep that caches
raw scores, so we run a separate sigma1 grid sweep here.

In [ ]:
from utls.sigma_fitting import compute_auroc_sparse, auc_to_dprime

def sweep_sigma1_raw_scores(
    sigma1_grid, sigma0, sigma2, t_step,
    noise_mode, metric, X0, name_to_idx,
    experiment_list, target_isis,
    n_mc=8, seed=0,
):
    """Sweep sigma1 and collect raw hit/FA scores for post-hoc analysis."""
    from utls.toy_experiments import infer_trial_isis
    seq_trial_isis = [infer_trial_isis(seq) for seq in experiment_list]
    results = []

    for gi, sigma1_val in enumerate(sigma1_grid):
        all_hits_by_isi = {isi: [] for isi in target_isis}
        all_fas = []

        for rep in range(n_mc):
            for si, seq in enumerate(experiment_list):
                t_isis = seq_trial_isis[si]
                run_out = run_experiment_scores(
                    sigma0=sigma0, sigma1=sigma1_val, sigma2=sigma2,
                    t_step=t_step, rate=0, noise_mode=noise_mode,
                    metric=metric, X0=X0, name_to_idx=name_to_idx,
                    experiment_list=[seq], debug=False,
                    seed=seed + gi * 100_000 + rep * 1000 + si,
                )
                h = np.asarray(run_out["hits"])
                f = np.asarray(run_out["fas"])
                if len(h) != len(t_isis):
                    continue
                for isi_val in target_isis:
                    mask = np.array(t_isis) == isi_val
                    all_hits_by_isi[isi_val].extend(h[mask].tolist())
                all_fas.extend(f.tolist())

        results.append({
            "sigma1": sigma1_val,
            "hit_scores_by_isi": {isi: np.array(v) for isi, v in all_hits_by_isi.items()},
            "fa_scores": np.array(all_fas),
        })
        print(f"  sigma1={sigma1_val:.4f}: {sum(len(v) for v in all_hits_by_isi.values())} hits, {len(all_fas)} fas")

    return results

print("sweep_sigma1_raw_scores defined.")

In [ ]:
# Run a single sweep with N=10 sequences, caching raw scores
N_FOR_NPOINTS = 10
rng_np = np.random.default_rng(seed=42 + N_FOR_NPOINTS)
np_indices = rng_np.choice(len(sigma1_all_exps), size=N_FOR_NPOINTS, replace=False)
np_exps = [sigma1_all_exps[i] for i in np_indices]

SIGMA1_GRID_FINE = np.exp(np.linspace(np.log(param_bounds["sigma1"][0]),
                                       np.log(param_bounds["sigma1"][1]), 15))

print(f"Running raw-score sweep: {len(SIGMA1_GRID_FINE)} sigma1 values, "
      f"{N_FOR_NPOINTS} sequences, N_MC={N_MC}")

raw_sweep = sweep_sigma1_raw_scores(
    sigma1_grid=SIGMA1_GRID_FINE,
    sigma0=sigma0_fitted,
    sigma2=sigma2_init,
    t_step=t_step,
    noise_mode=noise_mode,
    metric=metric,
    X0=X,
    name_to_idx=name_to_idx,
    experiment_list=np_exps,
    target_isis=SIGMA1_ISIS,
    n_mc=N_MC,
    seed=500_000,
)
print("Done.")

In [ ]:
# Recompute d' with varying n_points
N_POINTS_SWEEP = [4, 8, 12, 20, 50, 100, 500]

dprime_by_npoints = {}
for n_pts in N_POINTS_SWEEP:
    dp_by_isi = {isi: [] for isi in SIGMA1_ISIS}
    for r in raw_sweep:
        for isi_val in SIGMA1_ISIS:
            hits = r["hit_scores_by_isi"][isi_val]
            fas = r["fa_scores"]
            auroc = compute_auroc_sparse(hits, fas, n_points=n_pts)
            dp_by_isi[isi_val].append(auc_to_dprime(auroc, eps=1e-4))
    dprime_by_npoints[n_pts] = dp_by_isi

sigma1_vals = [r["sigma1"] for r in raw_sweep]
print(f"Recomputed d' for n_points = {N_POINTS_SWEEP}")

In [ ]:
import matplotlib.cm as cm

fig, axes = plt.subplots(1, len(SIGMA1_ISIS), figsize=(6 * len(SIGMA1_ISIS), 5),
                         sharey=True)

colors = cm.plasma(np.linspace(0.1, 0.9, len(N_POINTS_SWEEP)))

for ax, isi_val in zip(axes, SIGMA1_ISIS):
    for color, n_pts in zip(colors, N_POINTS_SWEEP):
        dp = dprime_by_npoints[n_pts][isi_val]
        ax.plot(sigma1_vals, dp, "o-", ms=3, color=color,
                label=f"n_pts={n_pts}", alpha=0.8)

    ax.axhline(sigma1_human[isi_val], ls=":", color="red", alpha=0.7, label="human d'")
    ax.set_xscale("log")
    ax.set_xlabel("sigma1")
    ax.set_title(f"ISI = {isi_val}")
    ax.grid(alpha=0.25)
    ax.legend(fontsize=7, frameon=False, ncol=2)

axes[0].set_ylabel("d'")
fig.suptitle("d' vs sigma1 — effect of n_points in sparse AUROC", y=1.03, fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
# Summary: max d' range across n_points for each ISI
print(f"{'n_points':>8s}", end="")
for isi_val in SIGMA1_ISIS:
    print(f" | {'ISI='+str(isi_val)+' max|dp|':>14s}", end="")
print()
print("-" * 60)

for n_pts in N_POINTS_SWEEP:
    print(f"{n_pts:>8d}", end="")
    for isi_val in SIGMA1_ISIS:
        dp = np.array(dprime_by_npoints[n_pts][isi_val])
        dp_range = np.nanmax(dp) - np.nanmin(dp)
        print(f" | {dp_range:>14.4f}", end="")
    print()

print("\nSmaller range = flatter curve = less sensitivity to sigma1")
print("Larger range = more dynamic range = better fitting resolution")